# FLUX Q8 双开 — 常驻版（模型装一次、循环出图）

与 `flux-portraits.ipynb` 的区别只有出图段：两个 worker 进程各钉一张卡，**主干模型只加载一次**，
然后按分到的序号列表循环出图，不再每批重装 12.7 GB 模型、也不再每批错峰 60 秒。

实测基线（旧版，每批 2 张）：编码 ~73s + 双开 ~274s ≈ **347s/批**。
常驻版预期：编码只做一次、加载+错峰只付一次，后续每张只剩纯生成 ~141s。

铁律不变：gguf 先于 diffusers 导入；GGUF 模型只 `.to(device)` 不转精度；
编码关梯度；下载走 Kaggle Secrets 的 HF_TOKEN（本地模板不带令牌）。


In [ ]:
# ===== 参数区（只改这里）=====
# 默认 4 张 = 旧版 2 批的量，专门用来同口径对比常驻版到底省多少。
PROMPTS = [
    "photorealistic portrait of a beautiful Chinese woman in her late 20s, elegant red qipao with floral embroidery, standing on an old Shanghai street at dusk, warm lantern light, cinematic, 85mm lens, ultra detailed",
    "photorealistic portrait of a beautiful Chinese woman in her early 30s, flowing white summer dress, standing on a beach at golden hour, sea breeze in her hair, ocean and sunset behind, ultra detailed",
    "photorealistic portrait of a Chinese woman pianist in an elegant dress, playing a grand piano in a sunlit music room, soft window light, cinematic, ultra detailed",
    "photorealistic portrait of a beautiful Chinese woman in a red coat, standing by an ancient temple in light snow, serene winter atmosphere, cinematic, ultra detailed",
]
SEEDS = [201, 202, 203, 204]   # 与 PROMPTS 一一对应
STEPS, GUIDANCE, W, H = 28, 3.5, 1024, 1024
STAGGER_S = 60                  # 只在两个 worker 首次启动之间错峰一次
assert len(PROMPTS) == len(SEEDS) and len(PROMPTS) >= 2


In [ ]:
# ===== 下载全部权重（逐项掐表）=====
import subprocess, sys, os, time, glob, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf==0.19.0"], check=True)
import gguf  # 先于 diffusers
import torch
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = tok
from huggingface_hub import snapshot_download, hf_hub_download
BASE = "/kaggle/working/flux_local"; os.makedirs(BASE, exist_ok=True)
T_INGRESS = time.time()
def timed(label, fn):
    t0 = time.time(); r = fn(); print(f"[计时] {label}: {time.time()-t0:.1f}s"); return r
timed("小件 CLIP/VAE/tokenizer/配置", lambda: snapshot_download(
    "black-forest-labs/FLUX.1-dev", local_dir=BASE, token=tok,
    allow_patterns=["text_encoder/**", "vae/**", "tokenizer/**", "tokenizer_2/**",
                    "scheduler/**", "model_index.json", "transformer/config.json",
                    "text_encoder_2/config.json", "text_encoder_2/generation_config.json"]))
for f in glob.glob(BASE + "/text_encoder_2/*.safetensors"): os.remove(f)
p = timed("T5 fp8", lambda: hf_hub_download("comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn.safetensors", local_dir="/kaggle/working/hf_t5"))
shutil.move(p, BASE + "/text_encoder_2/model.safetensors")  # move 而非 copy：T5 双份占盘会撑爆 19.5G 工作区（实测教训）
GGUF_PATH = timed("GGUF Q8_0", lambda: hf_hub_download("city96/FLUX.1-dev-gguf", "flux1-dev-Q8_0.gguf", local_dir=BASE))
DS = BASE
print(f"[计时] 全部下载总计: {time.time()-T_INGRESS:.1f}s")
print("DS:", DS); print(sorted(os.listdir(DS)))
print("GPUs:", torch.cuda.device_count())

In [ ]:
# ===== 编码（关梯度；编码器用完即卸）=====
from transformers import CLIPTextModel, CLIPTokenizer, T5EncoderModel, T5TokenizerFast
torch.set_grad_enabled(False)
tok1 = CLIPTokenizer.from_pretrained(DS, subfolder="tokenizer")
tok2 = T5TokenizerFast.from_pretrained(DS, subfolder="tokenizer_2")
t0 = time.time()
te1 = CLIPTextModel.from_pretrained(DS, subfolder="text_encoder", torch_dtype=torch.float16).to("cuda:0")
te2 = T5EncoderModel.from_pretrained(DS, subfolder="text_encoder_2", torch_dtype=torch.float16).to("cuda:0")
print(f"编码器加载: {time.time()-t0:.1f}s")
def encode(prompt):
    i1 = tok1(prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to("cuda:0")
    pooled = te1(i1).pooler_output
    i2 = tok2(prompt, padding="max_length", max_length=512, truncation=True, return_tensors="pt").input_ids.to("cuda:0")
    return te2(i2).last_hidden_state.cpu(), pooled.cpu()
os.makedirs("/kaggle/working/embeds", exist_ok=True)
for i, p in enumerate(PROMPTS):
    emb, pooled = encode(p)
    torch.save({"emb": emb, "pooled": pooled, "seed": SEEDS[i]}, f"/kaggle/working/embeds/emb_{i}.pt")
    print(i, emb.shape, pooled.shape)
del te1, te2; torch.cuda.empty_cache()
print(f"编码阶段完成，总计 {time.time()-t0:.1f}s（含加载）")

In [ ]:
# ===== 常驻 worker 脚本（加载一次，循环出图）=====
worker = r"""
import sys, time, json, gc
import gguf
import torch
from diffusers import FluxTransformer2DModel, FluxPipeline, AutoencoderKL, FlowMatchEulerDiscreteScheduler, GGUFQuantizationConfig

ds, gguf_path = sys.argv[1], sys.argv[2]
indices = [int(x) for x in sys.argv[3].split(",") if x != ""]
steps, guidance, w, h = int(sys.argv[4]), float(sys.argv[5]), int(sys.argv[6]), int(sys.argv[7])

t0 = time.time()
transformer = FluxTransformer2DModel.from_single_file(gguf_path, config=f"{ds}/transformer", quantization_config=GGUFQuantizationConfig(compute_dtype=torch.float16))
transformer = transformer.to("cuda:0")  # 量化模型不可转精度
vae = AutoencoderKL.from_pretrained(ds, subfolder="vae", torch_dtype=torch.float16).to("cuda:0")
sched = FlowMatchEulerDiscreteScheduler.from_pretrained(ds, subfolder="scheduler")
pipe = FluxPipeline(scheduler=sched, vae=vae, text_encoder=None, tokenizer=None, text_encoder_2=None, tokenizer_2=None, transformer=transformer)
t_load = time.time() - t0
print(json.dumps({"event": "loaded", "load_s": round(t_load, 1), "indices": indices}), flush=True)

for idx in indices:
    try:
        d = torch.load(f"/kaggle/working/embeds/emb_{idx}.pt")
        g = torch.Generator(device="cuda:0").manual_seed(int(d["seed"]))
        t1 = time.time()
        img = pipe(prompt_embeds=d["emb"].to("cuda:0", torch.float16), pooled_prompt_embeds=d["pooled"].to("cuda:0", torch.float16),
                   width=w, height=h, num_inference_steps=steps, guidance_scale=guidance, generator=g).images[0]
        img.save(f"/kaggle/working/out_{idx}.png")
        print(json.dumps({"event": "image", "idx": idx, "gen_plus_decode_s": round(time.time() - t1, 1),
                          "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}), flush=True)
    except Exception as e:
        print(json.dumps({"event": "error", "idx": idx, "error": repr(e)}), flush=True)
    finally:
        gc.collect(); torch.cuda.empty_cache()  # 每图后归还预留显存，防常驻碎片累积（实测 OOM 修复）
print(json.dumps({"event": "done", "load_s": round(t_load, 1), "count": len(indices)}), flush=True)
"""
open("/kaggle/working/worker_persistent.py", "w").write(worker)
print("worker_persistent.py 已写入")


In [ ]:
# ===== 双卡常驻启动：序号轮流分卡，错峰只付一次 =====
import subprocess, sys, os, time
assign = {0: [], 1: []}
for i in range(len(PROMPTS)):
    assign[i % 2].append(i)
print("分卡:", assign)
procs = []
t0 = time.time()
for gpu in (0, 1):
    if not assign[gpu]:
        continue
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")  # 抗显存碎片（实测 OOM 修复）
    args = [sys.executable, "/kaggle/working/worker_persistent.py", DS, GGUF_PATH,
            ",".join(str(i) for i in assign[gpu]), str(STEPS), str(GUIDANCE), str(W), str(H)]
    procs.append(subprocess.Popen(args, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True))
    if gpu == 0 and assign[1]:
        time.sleep(STAGGER_S)  # 只错峰这一次
for p in procs:
    print(p.communicate()[0])
wall = time.time() - t0
n = len(PROMPTS)
print(f"常驻版墙钟: {wall:.1f}s | 共 {n} 张 | 每图平均: {wall/n:.1f}s | 折合每批(2张): {wall/(n/2):.1f}s")
print(f"旧版基线: 每批约 347s（编码73+双开274，未含重复下载）；同量旧版约 {347*(n/2):.0f}s")


In [ ]:
from IPython.display import Image, display
import os
for i in range(len(PROMPTS)):
    f = f"/kaggle/working/out_{i}.png"
    if os.path.exists(f):
        display(Image(filename=f))
    else:
        print(f"out_{i}.png 缺失，看上面 worker 日志的 error 事件")
